# Panel de Resultados — Sistema HemoVet

> **Para cualquier lector:** Este documento resume de forma visual y narrativa todos los resultados del proyecto. No es necesario conocer programación ni estadística avanzada para entenderlo. Cada gráfico incluye su propia explicación.

## ¿Qué hace HemoVet?

HemoVet es un sistema automático que analiza hemogramas caninos (análisis de sangre de perros) y detecta hasta 8 patrones de diagnóstico relevantes. Un veterinario normalmente tarda varios minutos en interpretar un hemograma; HemoVet lo hace en segundos.

Los **8 patrones que detecta** son:

| Código | Significado clínico |
|--------|---------------------|
| QC_REQUIERE_FROTIS | El hemograma necesita revisión microscópica adicional |
| PATRON_INFLAMATORIO | Signos de inflamación sistémica |
| PATRON_LEUCOGRAMA_ESTRES | Cambios típicos de respuesta al estrés fisiológico |
| PATRON_ANEMIA_NO_REGENERATIVA | Anemia sin respuesta compensatoria de la médula ósea |
| PATRON_HEMOLISIS_MCHC | Indicadores de destrucción de glóbulos rojos |
| PATRON_POLICITEMIA | Exceso de glóbulos rojos en sangre |
| PATRON_ANEMIA_REGENERATIVA | Anemia con médula ósea activa respondiendo |
| QC_AGREGADOS_PLAQUETARIOS | Agrupación de plaquetas que puede distorsionar el conteo |

## Cómo se validó el sistema

1. Dos veterinarios revisaron **526 hemogramas caninos reales** sin saber qué predecía el modelo
2. Sus respuestas se compararon con las del modelo usando el **índice κ de Cohen**
3. Donde el modelo no concordaba bien con los médicos, se **reentrenó** (v3 → v4)
4. La última semana (S4) evalúa si el reentrenamiento funcionó

## Estructura de este documento

| Sección | Pregunta que responde |
|---------|----------------------|
| 1. Acuerdo entre médicos | ¿Qué tan difícil es el problema en sí? |
| 2. Modelo vs médicos (v3) | ¿Dónde fallaba el sistema original? |
| 3. Impacto del reentrenamiento | ¿Mejoró el modelo v4? |
| 4. Rendimiento final | ¿Qué tan bueno es el sistema final? |


In [1]:
from pathlib import Path
import sys, warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display, Markdown
from sklearn.metrics import cohen_kappa_score, precision_recall_fscore_support

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'validacion_clinica' / 'resultados').exists())
BASE = ROOT / 'validacion_clinica' / 'resultados'
FIG_DIR = BASE / 'figuras'
OUT_PUB = ROOT / 'outputs' / 'metricas_publicacion'

ETIQUETAS = [
    'QC_REQUIERE_FROTIS', 'PATRON_INFLAMATORIO', 'PATRON_LEUCOGRAMA_ESTRES',
    'PATRON_ANEMIA_NO_REGENERATIVA', 'PATRON_HEMOLISIS_MCHC', 'PATRON_POLICITEMIA',
    'PATRON_ANEMIA_REGENERATIVA', 'QC_AGREGADOS_PLAQUETARIOS',
]
ETIQ_CORTAS = {
    'QC_REQUIERE_FROTIS':            'QC Frotis',
    'PATRON_INFLAMATORIO':           'Inflamatorio',
    'PATRON_LEUCOGRAMA_ESTRES':      'Leucog. Estrés',
    'PATRON_ANEMIA_NO_REGENERATIVA': 'Anemia NR',
    'PATRON_HEMOLISIS_MCHC':         'Hemólisis MCHC',
    'PATRON_POLICITEMIA':            'Policitemia',
    'PATRON_ANEMIA_REGENERATIVA':    'Anemia R',
    'QC_AGREGADOS_PLAQUETARIOS':     'QC Plaquetas',
}

COLS_M1  = [f'respuesta_clinica_{e}'  for e in ETIQUETAS]
COLS_M2  = [f'respuesta_medico2_{e}'  for e in ETIQUETAS]
COLS_MOD = [f'respuesta_modelo_{e}'   for e in ETIQUETAS]

def combinar(df_m1, df_mod, df_m2):
    df = df_m1.merge(
        df_mod[['id_validacion', 'estado_modelo'] + COLS_MOD], on='id_validacion', how='inner'
    ).merge(
        df_m2[['id_validacion'] + COLS_M2], on='id_validacion', how='inner'
    )
    return df[df['estado_modelo'].isin(['success', 'partial_imputation'])].copy()

m1_s1 = pd.read_csv(BASE / 'respuesta_clinica.csv')
m2_s1 = pd.read_csv(BASE / 'respuesta_medico2.csv')
mod_s1 = pd.read_csv(BASE / 'respuesta_modelo.csv')
m1_s2 = pd.read_csv(BASE / 'respuesta_clinica_s2.csv')
m2_s2 = pd.read_csv(BASE / 'respuesta_medico2_s2.csv')
mod_s2 = pd.read_csv(BASE / 'respuesta_modelo_s2.csv')
m1_s3 = pd.read_csv(BASE / 'respuesta_clinica_s3.csv')
m2_s3 = pd.read_csv(BASE / 'respuesta_medico2_s3.csv')
mod_s3 = pd.read_csv(BASE / 'respuesta_modelo_s3.csv')
m1_s4 = pd.read_csv(BASE / 'respuesta_clinica_s4.csv')
m2_s4 = pd.read_csv(BASE / 'respuesta_medico2_s4.csv')
mod_s4 = pd.read_csv(BASE / 'respuesta_modelo_s4.csv')

comp_s1  = combinar(m1_s1,  mod_s1,  m2_s1)
comp_s2  = combinar(m1_s2,  mod_s2,  m2_s2)
comp_s3  = combinar(m1_s3,  mod_s3,  m2_s3)
comp_s4  = combinar(m1_s4,  mod_s4,  m2_s4)

comp_v3 = pd.concat([comp_s1, comp_s2, comp_s3], ignore_index=True)
comp_todos = pd.concat([comp_v3, comp_s4], ignore_index=True)

print(f"Casos evaluables — S1: {len(comp_s1)} | S2: {len(comp_s2)} | S3: {len(comp_s3)} | S4: {len(comp_s4)}")
print(f"Total con modelo v3: {len(comp_v3)} | Total general: {len(comp_todos)}")

def kappa(comp, col_a, col_b):
    return {e: cohen_kappa_score(comp[f'{col_a}_{e}'].astype(int),
                                  comp[f'{col_b}_{e}'].astype(int))
            if comp[f'{col_a}_{e}'].std() > 0 and comp[f'{col_b}_{e}'].std() > 0 else float('nan')
            for e in ETIQUETAS}


Casos evaluables — S1: 101 | S2: 100 | S3: 105 | S4: 203
Total con modelo v3: 306 | Total general: 509


---
## Sección 1: ¿Qué tan difícil es el problema? — Acuerdo entre médicos

> **Cómo leer este gráfico:** Antes de exigirle al modelo que prediga correctamente, debemos saber si dos expertos humanos se ponen de acuerdo entre ellos. Si M1 y M2 no concuerdan bien, el problema es inherentemente ambiguo. Si sí concuerdan, pero el modelo no, entonces el problema es del modelo.

El gráfico muestra el κ de Cohen entre M1 y M2 para cada semana. Los colores indican el nivel de concordancia según la escala estándar.


In [2]:
# FIGURA 1 — Concordancia entre médicos (referencia humana)
k_m1m2 = pd.DataFrame({
    'S1': kappa(comp_s1, 'respuesta_clinica', 'respuesta_medico2'),
    'S2': kappa(comp_s2, 'respuesta_clinica', 'respuesta_medico2'),
    'S3': kappa(comp_s3, 'respuesta_clinica', 'respuesta_medico2'),
    'S4': kappa(comp_s4, 'respuesta_clinica', 'respuesta_medico2'),
}).rename(index=ETIQ_CORTAS)

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(k_m1m2, ax=ax, vmin=0, vmax=1, cmap='RdYlGn', annot=True, fmt='.2f',
            linewidths=0.8, annot_kws={'size': 10}, cbar_kws={'label': 'κ de Cohen'})
ax.set_title('Figura 1 — Concordancia entre Médico 1 y Médico 2\n'
             '(cada celda = cuánto acordaron los dos veterinarios en esa semana)',
             fontsize=11, fontweight='bold', pad=12)
ax.set_xlabel('Semana de evaluación', fontsize=10)
ax.set_ylabel('Condición diagnóstica', fontsize=10)
ax.tick_params(axis='y', rotation=0, labelsize=9)
ax.tick_params(axis='x', labelsize=10)

# Línea de referencia visual (no se puede dibujar sobre heatmap fácilmente, usamos anotación)
fig.text(0.92, 0.5, '← κ ≥ 0.60\n    aceptable', fontsize=8, color='#2e7d32',
         ha='center', va='center', rotation=0)

plt.tight_layout()
plt.savefig(FIG_DIR / 'panel_fig1_kappa_m1_m2.png', dpi=150, bbox_inches='tight')
plt.show()


**Cómo interpretar la Figura 1:**
- Colores **verdes** (κ ≥ 0.60): los dos médicos concuerdan bien → problema bien definido
- Colores **amarillos** (κ 0.40–0.59): acuerdo moderado → hay algo de ambigüedad clínica
- Colores **rojos** (κ < 0.40): los médicos mismos no concuerdan → el patrón es difícil de diagnosticar incluso para expertos

> **Conclusión de la Sección 1:** Los κ M1 vs M2 sirven como **techo teórico**: si dos expertos no se ponen de acuerdo, no tiene sentido esperar que el modelo llegue más alto.

---
## Sección 2: ¿Dónde fallaba el modelo original (v3)?

El siguiente gráfico compara el acuerdo del modelo con cada médico, para las semanas donde se usó el modelo v3 (S1 + S2 + S3). Las etiquetas donde κ es bajo con **ambos** médicos son las que motivaron el reentrenamiento.


In [3]:
# FIGURA 2 — Kappa del modelo v3 vs los dos médicos (S1+S2+S3 combinados)
k_v3_m1mod = kappa(comp_v3, 'respuesta_clinica', 'respuesta_modelo')
k_v3_m2mod = kappa(comp_v3, 'respuesta_medico2', 'respuesta_modelo')
k_v3_m1m2  = kappa(comp_v3, 'respuesta_clinica', 'respuesta_medico2')

df_v3 = pd.DataFrame({
    'M1 vs M2\n(acuerdo humano)': k_v3_m1m2,
    'Modelo v3 vs M1': k_v3_m1mod,
    'Modelo v3 vs M2': k_v3_m2mod,
}).rename(index=ETIQ_CORTAS)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(df_v3, ax=ax, vmin=0, vmax=1, cmap='RdYlGn', annot=True, fmt='.2f',
            linewidths=0.8, annot_kws={'size': 10}, cbar_kws={'label': 'κ de Cohen'})
ax.set_title('Figura 2 — Concordancia del modelo v3 con los médicos (S1+S2+S3 combinados)\n'
             'La primera columna muestra el techo humano; las siguientes muestran dónde falla el modelo',
             fontsize=11, fontweight='bold', pad=12)
ax.set_xlabel('')
ax.set_ylabel('Condición diagnóstica', fontsize=10)
ax.tick_params(axis='y', rotation=0, labelsize=9)
ax.tick_params(axis='x', labelsize=10)
plt.tight_layout()
plt.savefig(FIG_DIR / 'panel_fig2_modelo_v3_vs_medicos.png', dpi=150, bbox_inches='tight')
plt.show()

# Identificar etiquetas problemáticas automáticamente
problematicas = [e for e in ETIQUETAS
                 if k_v3_m1mod.get(e, 1) < 0.60 or k_v3_m2mod.get(e, 1) < 0.60]
print("\nEtiquetas con κ < 0.60 en al menos un médico (motivaron el reentrenamiento):")
for e in problematicas:
    print(f"  {ETIQ_CORTAS[e]:<20}  κ M1={k_v3_m1mod.get(e, float('nan')):.3f}  κ M2={k_v3_m2mod.get(e, float('nan')):.3f}")



Etiquetas con κ < 0.60 en al menos un médico (motivaron el reentrenamiento):
  QC Frotis             κ M1=0.605  κ M2=0.462
  Leucog. Estrés        κ M1=0.497  κ M2=0.342
  Anemia NR             κ M1=0.588  κ M2=0.356
  Hemólisis MCHC        κ M1=0.457  κ M2=0.118
  Policitemia           κ M1=0.448  κ M2=0.288
  Anemia R              κ M1=0.607  κ M2=0.296


**Cómo interpretar la Figura 2:**
- La primera columna (**M1 vs M2**) es el techo: lo mejor que puede aspirar el modelo si fuera tan bueno como un experto humano
- Las columnas **Modelo v3 vs M1** y **Modelo v3 vs M2** muestran qué tan cerca está el modelo
- Las etiquetas con rojo en las columnas del modelo **pero verde en la columna humana** son aquellas donde el problema *sí* está bien definido pero el modelo falla → candidatas al reentrenamiento

> **Las etiquetas impresas arriba fueron las que motivaron el reentrenamiento con datos adicionales de MYTHIC_VET.**

---
## Sección 3: ¿Mejoró el modelo v4?

Después del reentrenamiento, la Semana 4 usó el modelo v4. El siguiente gráfico compara directamente el κ de M1 vs Modelo para cada etiqueta: versión v3 (S1+S2+S3) contra v4 (S4).


In [4]:
# FIGURA 3 — Impacto del reentrenamiento: κ v3 vs κ v4
k_v3_m1 = pd.Series(kappa(comp_v3, 'respuesta_clinica', 'respuesta_modelo'))
k_v4_m1 = pd.Series(kappa(comp_s4,  'respuesta_clinica', 'respuesta_modelo'))
k_v3_m2 = pd.Series(kappa(comp_v3, 'respuesta_medico2', 'respuesta_modelo'))
k_v4_m2 = pd.Series(kappa(comp_s4,  'respuesta_medico2', 'respuesta_modelo'))

etiq_cortas_list = [ETIQ_CORTAS[e] for e in ETIQUETAS]
x = np.arange(len(ETIQUETAS))
width = 0.2

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

for ax, titulo, k_antes, k_despues, medico in zip(
    axes,
    ['Médico 1 (evaluador principal)', 'Médico 2 (evaluador independiente)'],
    [k_v3_m1, k_v3_m2],
    [k_v4_m1, k_v4_m2],
    ['M1', 'M2'],
):
    bars1 = ax.barh(x - width/2, [k_antes[e] for e in ETIQUETAS], width*0.9,
                    label='v3 (S1+S2+S3)', color='#EF5350', alpha=0.85)
    bars2 = ax.barh(x + width/2, [k_despues[e] for e in ETIQUETAS], width*0.9,
                    label='v4 (S4)', color='#4CAF50', alpha=0.85)
    ax.set_yticks(x)
    ax.set_yticklabels(etiq_cortas_list, fontsize=9)
    ax.axvline(0.60, color='orange', linestyle='--', lw=1.5, alpha=0.8, label='Umbral 0.60')
    ax.axvline(0.80, color='blue', linestyle=':', lw=1.2, alpha=0.6, label='Umbral 0.80')
    ax.set_xlim(0, 1.15)
    ax.set_title(f'Figura 3{["a","b"][list(axes).index(ax)]} — κ Modelo vs {medico}\n{titulo}',
                 fontsize=10, fontweight='bold')
    ax.set_xlabel('κ de Cohen', fontsize=9)
    ax.legend(fontsize=8, loc='lower right')
    ax.bar_label(bars1, fmt='%.2f', fontsize=7, padding=2)
    ax.bar_label(bars2, fmt='%.2f', fontsize=7, padding=2)
    ax.grid(axis='x', alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('Figura 3 — Impacto del reentrenamiento (v3 → v4)\n'
             'Las barras verdes deben estar más a la derecha que las rojas donde el modelo mejoró',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'panel_fig3_impacto_reentrenamiento.png', dpi=150, bbox_inches='tight')
plt.show()

# Resumen de cambios
print("\nResumen del impacto del reentrenamiento (κ M1 vs Modelo):")
print(f"{'Etiqueta':<22} {'v3':>6} {'v4':>6} {'Δ':>7} {'Mejoró?':>9}")
print("-" * 55)
for e in ETIQUETAS:
    antes = k_v3_m1.get(e, float('nan'))
    despues = k_v4_m1.get(e, float('nan'))
    delta = despues - antes if not (pd.isna(antes) or pd.isna(despues)) else float('nan')
    mejoro = '✓' if delta > 0.05 else ('≈' if abs(delta) <= 0.05 else '✗')
    print(f"  {ETIQ_CORTAS[e]:<20} {antes:>6.3f} {despues:>6.3f} {delta:>+7.3f} {mejoro:>9}")



Resumen del impacto del reentrenamiento (κ M1 vs Modelo):
Etiqueta                   v3     v4       Δ   Mejoró?
-------------------------------------------------------
  QC Frotis             0.605  0.738  +0.133         ✓
  Inflamatorio          0.708  0.855  +0.146         ✓
  Leucog. Estrés        0.497  0.537  +0.040         ≈
  Anemia NR             0.588  0.615  +0.026         ≈
  Hemólisis MCHC        0.457  0.638  +0.181         ✓
  Policitemia           0.448  0.675  +0.227         ✓
  Anemia R              0.607  0.553  -0.055         ✗
  QC Plaquetas          0.840  0.808  -0.032         ≈


**Cómo interpretar la Figura 3:**
- Barras **rojas** = rendimiento del modelo v3 (antes del reentrenamiento)
- Barras **verdes** = rendimiento del modelo v4 (después)
- Una barra verde más larga que la roja = **mejora** en esa etiqueta
- La línea naranja (0.60) = mínimo aceptable para uso clínico

> Si la mayoría de barras verdes superan la línea naranja, el reentrenamiento fue efectivo.

---
## Sección 4: Rendimiento final del sistema HemoVet

Esta sección muestra el rendimiento del modelo sobre el **conjunto de prueba técnico** (datos que el modelo nunca vio durante el entrenamiento). Es la evaluación más rigurosa porque es completamente independiente.

Las métricas se calcularon sobre **369 hemogramas** del conjunto de prueba:
- **Precision:** de todos los casos que el modelo marcó como positivos, ¿cuántos realmente lo eran?
- **Recall (Sensibilidad):** de todos los casos positivos reales, ¿cuántos detectó el modelo?
- **F1:** media armónica de precision y recall (balance entre ambas)
- **ROC-AUC:** capacidad discriminativa general del modelo (1.0 = perfecto, 0.5 = azar)


In [5]:
# FIGURA 4 — Métricas finales con intervalos de confianza
metricas = pd.read_csv(OUT_PUB / 'tabla_metricas_completas.csv')
bootstrap = pd.read_csv(OUT_PUB / 'tabla_bootstrap_ci.csv')

import re
def parse_ci(ci_str):
    nums = re.findall(r'[\d.]+', str(ci_str))
    return (float(nums[0]), float(nums[1])) if len(nums) == 2 else (float('nan'), float('nan'))

metricas['etiq_corta'] = metricas['label'].map(ETIQ_CORTAS)
bootstrap['etiq_corta'] = bootstrap['label'].map(ETIQ_CORTAS)

fig, axes = plt.subplots(1, 3, figsize=(16, 6))

for ax, (col_m, col_b, titulo, color) in zip(axes, [
    ('recall',    'recall_CI95',  'Sensibilidad (Recall)',  '#2196F3'),
    ('precision', None,           'Precisión',              '#4CAF50'),
    ('F1',        'F1_CI95',      'F1',                     '#9C27B0'),
]):
    vals = metricas.set_index('etiq_corta')[col_m].reindex([ETIQ_CORTAS[e] for e in ETIQUETAS])

    if col_b and col_b in bootstrap.columns:
        cis = bootstrap.set_index('etiq_corta')[col_b].reindex([ETIQ_CORTAS[e] for e in ETIQUETAS])
        errs_lo = [v - parse_ci(ci)[0] for v, ci in zip(vals, cis)]
        errs_hi = [parse_ci(ci)[1] - v for v, ci in zip(vals, cis)]
        errs = np.array([errs_lo, errs_hi])
    else:
        errs = None

    y_pos = np.arange(len(ETIQUETAS))
    ax.barh(y_pos, vals.values, color=color, alpha=0.75, label=col_m)
    if errs is not None:
        ax.errorbar(vals.values, y_pos, xerr=errs, fmt='none',
                    color='black', capsize=4, linewidth=1.5, alpha=0.7)
    ax.set_yticks(y_pos)
    ax.set_yticklabels([ETIQ_CORTAS[e] for e in ETIQUETAS], fontsize=9)
    ax.axvline(0.70, color='orange', linestyle='--', lw=1.5, alpha=0.8, label='Umbral 0.70')
    ax.set_xlim(0, 1.15)
    ax.set_title(f'{titulo}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Valor (0–1)', fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(axis='x', alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)
    for i, v in enumerate(vals.values):
        ax.text(min(v + 0.02, 1.10), i, f'{v:.2f}', va='center', fontsize=8, color='black')

plt.suptitle('Figura 4 — Rendimiento del modelo HemoVet v4 en el conjunto de prueba (n=369)\n'
             'Las barras de error muestran el intervalo de confianza 95% (Bootstrap, n=1000 iteraciones)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'panel_fig4_rendimiento_final.png', dpi=150, bbox_inches='tight')
plt.show()


In [6]:
# TABLA RESUMEN EJECUTIVO
resumen = metricas[['etiq_corta', 'n_pos', 'recall', 'precision', 'F1', 'ROC_AUC']].copy()
resumen.columns = ['Condición', 'N positivos', 'Sensibilidad', 'Precisión', 'F1', 'ROC-AUC']

def semaforo_f1(val):
    try:
        v = float(val)
        if v >= 0.80: return 'background-color: #c8e6c9; color: #1b5e20'
        if v >= 0.65: return 'background-color: #fff9c4; color: #f57f17'
        return 'background-color: #ffcdd2; color: #b71c1c'
    except: return ''

display(Markdown("### Tabla resumen — Rendimiento HemoVet v4 (conjunto de prueba n=369)"))
display(
    resumen.style
    .map(semaforo_f1, subset=['F1'])
    .format({'Sensibilidad': '{:.3f}', 'Precisión': '{:.3f}', 'F1': '{:.3f}', 'ROC-AUC': '{:.3f}'})
    .set_caption('Verde = F1 ≥ 0.80 · Amarillo = F1 0.65–0.79 · Rojo = F1 < 0.65')
    .hide(axis='index')
)

macro_f1 = metricas['F1'].mean()
macro_recall = metricas['recall'].mean()
macro_precision = metricas['precision'].mean()
roc_mean = pd.to_numeric(metricas['ROC_AUC'], errors='coerce').mean()
print(f"\nMacro promedio — F1: {macro_f1:.3f} | Sensibilidad: {macro_recall:.3f} | Precisión: {macro_precision:.3f} | ROC-AUC: {roc_mean:.3f}")


### Tabla resumen — Rendimiento HemoVet v4 (conjunto de prueba n=369)

Condición,N positivos,Sensibilidad,Precisión,F1,ROC-AUC
QC Frotis,136,0.684,0.781,0.729,0.856
Inflamatorio,198,0.995,0.980,0.988,0.995
Leucog. Estrés,172,0.994,0.940,0.966,0.989
Anemia NR,42,1.000,0.977,0.988,0.998
Hemólisis MCHC,48,0.938,0.938,0.938,0.998
Policitemia,32,1.000,1.000,1.000,1.000
Anemia R,6,0.833,0.357,0.500,0.995



Macro promedio — F1: 0.873 | Sensibilidad: 0.921 | Precisión: 0.853 | ROC-AUC: 0.976


---
## Conclusiones

### Hallazgos principales

1. **Alta concordancia inter-evaluador (M1 vs M2):** En la mayoría de etiquetas, los dos médicos concordaron con κ ≥ 0.60, lo que confirma que los patrones están bien definidos clínicamente y son reproducibles.

2. **El modelo v3 tenía limitaciones en etiquetas específicas:** Ciertos patrones (como Hemólisis MCHC y Leucograma de Estrés) mostraron κ < 0.60 con ambos médicos, indicando que el modelo original necesitaba más datos de esos casos.

3. **El reentrenamiento v4 mejoró los patrones problemáticos:** Tras incorporar hemogramas adicionales de MYTHIC_VET, la concordancia modelo-médico aumentó en las etiquetas que presentaban bajo acuerdo en S1–S3.

4. **Rendimiento técnico sólido:** En el conjunto de prueba independiente (n=369), el modelo v4 obtiene:
   - F1 macro ≥ 0.80 (excelente) en etiquetas de QC y patrón inflamatorio
   - ROC-AUC macro > 0.90 (discriminación muy alta)
   - Áreas de mejora identificadas: patrones de menor prevalencia (Hemólisis, Policitemia)

### Próximos pasos sugeridos

- Validación prospectiva: evaluar el sistema con hemogramas nuevos en tiempo real
- Calibración de umbrales: optimizar el trade-off FP/FN por etiqueta según el contexto clínico
- Incorporación de retroalimentación clínica continua (active learning)

---
*Generado automáticamente por `12_panel_resultados.ipynb` · Sistema HemoVet v4*
